In [1]:
%cd ../.
import os, sys
sys.path.insert(0, os.path.abspath('../Scripts'))

/home/gtamo/MS_ML


In [2]:
%load_ext autoreload
%autoreload 2

import re
import os
import pandas as pd
import numpy as np
import py3Dmol
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import csv
import pickle

from pathlib import Path
from rdkit import Chem
from rdkit.Chem import AllChem,rdFMCS
# import prolif as plf
from glob import glob
import meeko
import subprocess as sub
# from vina import Vina
import time
from tqdm import tqdm
tqdm.pandas()
import importlib
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.dummy import DummyClassifier
from sklearn.manifold import TSNE
from xgboost import XGBClassifier,XGBRegressor
from openTSNE import TSNE as oTSNE        # pip install openTSNE
import seaborn as sns
from scipy import stats
import csv, contextlib, threading, joblib
import joblib
from joblib import Parallel, delayed
from datetime import date

# user defined modules
import Rdkit_tools as rdkit_tools
importlib.reload(rdkit_tools)
import Molecule as M
import ML_Reg as ML_Reg
import ML_Class as ML_Class
from MolViz3D import MolViz3D
import Statistics_tools as stats_tools
import python.functions as fn
# from tdc.multi_pred import DTI

## 0. Imports

In [3]:
## params — single source of truth in config/config.yaml.
## Loaded as a `config` namespace AND injected as globals, so both
## `config.RAW_PROTEOMICS_PATH` and bare `RAW_PROTEOMICS_PATH` work.
import yaml
from types import SimpleNamespace
with open('config/config.yaml') as _f:
    _cfg = yaml.safe_load(_f)
config = SimpleNamespace(**_cfg)
globals().update(_cfg)
print(f'> loaded {len(_cfg)} params from config/config.yaml')


> loaded 33 params from config/config.yaml


### 20260528 - Prioritizing virtual & library compounds vs PCSK9
We will use the 20260513 trained models to proritize compounds coming from a virtual enumeration and unscreen compounds from our library

In [ ]:
## get enamine virtual enum - convert sdf to smiles and extract relevant fields/columns
enum_df = rdkit_tools.get_smiles_df_from_enum(ENAMINE_20260513)
enum_df['compound']  = enum_df['R1_Code'] + '_' + enum_df['R2_Code']
enum_df['filename'] = Path(ENAMINE_20260513).stem
enum_df['smiles'] = enum_df['smiles'].progress_apply(lambda x: rdkit_tools.convert_smiles_to_canonical(x))
enum_df.head(1)

In [ ]:
## get unscreened library:
lib = pd.read_csv(PX_SCREEN_LIB).rename(columns={'Molecule Name':'compound','SMILES':'smiles','Lib ID':'lib_id','Px_screened_source':'source'})
lib['smiles'] = lib['smiles'].progress_apply(lambda x: rdkit_tools.convert_smiles_to_canonical(x))
print(lib['Px_screened_anywhere'].unique())
lib['Px_sreened'] = 0
lib.loc[lib['Px_screened_anywhere']=='yes','Px_sreened'] = 1
lib = lib[['compound','smiles','lib_id','source','Px_sreened']]

# select uncreened compounds from enamine 4 lib id
# lib = lib[ (lib['lib_id']=='Enamine 4') & (lib['Px_sreened']==0) ]
# print(lib.shape)
# lib.head(1)

# get all the screened compounds:
Px_screened_smiles = list(lib[lib['Px_sreened']==1]['smiles'])
len(Px_screened_smiles)

In [ ]:
## remove anything that could have been screened from the enumeration:
print(f"before {enum_df.shape}")
enum_df = enum_df[~enum_df['smiles'].isin(Px_screened_smiles)]
print(f"after {enum_df.shape}")

In [ ]:
## prioritize compounds:
## Implementation now lives in Scripts/ML_Reg.py (ML_Reg.MLReg_prioritize_compounds).
## The function loads a joblib bundle, computes H236 features on `ori_data`, and
## returns the top-N most-active compounds. See its docstring for details.
model_path = 'output/ML/trained_models/20260513/PCSK9_RF_H236.joblib'

pred_df = ML_Reg.MLReg_prioritize_compounds(
    ori_data=enum_df,
    model=model_path,
    top=300,
    features_n=None,        # use the bundle's saved featurizer name
)


In [ ]:
test_outpath = DROPBOX_ML+'predictions/20260518_enum_NAr-pyrimidine_Enamine_4_pred.sdf'
pred_df

## Sanity check: SMILES intersection between the written SDF and pred_df
from rdkit import Chem

# 1) Read back what we wrote
out_df = rdkit_tools.get_smiles_df_from_enum(test_outpath)

# 2) Re-canonicalize both sides defensively — different code paths
#    (MolToSmiles in get_smiles_df_from_enum vs convert_smiles_to_canonical
#    upstream of pred_df) usually agree, but a one-pass re-canonicalisation
#    eliminates any kekulization / stereo-perception drift.
def _canon(smi):
    if not isinstance(smi, str) or not smi:
        return None
    m = Chem.MolFromSmiles(smi)
    return Chem.MolToSmiles(m) if m is not None else None

out_smis  = set(out_df['smiles'].map(_canon).dropna())
pred_smis = set(pred_df['smiles'].map(_canon).dropna())

inter      = out_smis & pred_smis
only_in_out  = out_smis - pred_smis
only_in_pred = pred_smis - out_smis

print(f'SDF written         : {len(out_df):,} rows  ({len(out_smis):,} unique canonical SMILES)')
print(f'pred_df             : {len(pred_df):,} rows ({len(pred_smis):,} unique canonical SMILES)')
print(f'∩ (intersection)    : {len(inter):,}')
print(f'in SDF, NOT in pred : {len(only_in_out):,}')
print(f'in pred, NOT in SDF : {len(only_in_pred):,}')




In [ ]:
## write sdf in dropbox folder:
if pred_df.shape[0] > 0:
    n = rdkit_tools.write_filtered_enum_sdf(
        pred_df,
        source_dir=DROPBOX_ML+'virtual libraries', # exact path to sdf file which let's copy verbatim the stereochemistry and attributes
        out_path=DROPBOX_ML+'predictions/20260518_enum_NAr-pyrimidine_Enamine_4_pred.sdf',
        pred_col='predicted_label',
)

#### Prioritize 150 compounds from 114 virtual enumeration

In [ ]:
## get 114 virtual enumerations:
enumpath = 'data/enumeration/20260427/'
enum_fs = glob(enumpath+'*')

# get a unified file for prediction:
# combined_sdf, n_parts = rdkit_tools.combine_sdfs(enumpath, '20260518_114K_enum.sdf')
enum114_df = rdkit_tools.get_smiles_df_from_enum_dir(enumpath, v=True)
print(f'> parsed {enum114_df["filename"].nunique()} SDFs, {len(enum114_df):,} rows total')


# get smiles from this:
enum114_df['compound']  = enum114_df['R1_Code'] + '_' + enum114_df['R2_Code']
enum114_df['smiles'] = enum114_df['smiles'].progress_apply(lambda x: rdkit_tools.convert_smiles_to_canonical(x))
enum114_df = enum114_df.drop_duplicates('smiles').reset_index(drop=True)

## remove compounds already present in other sets:
cm2rm = list(enum_df['smiles']) + Px_screened_smiles
print(f"before {enum114_df.shape}")
enum114_df = enum114_df[~enum114_df['smiles'].isin(cm2rm)]
print(f"before {enum114_df.shape}")


In [ ]:
## prioritize compounds:
pred114_df = ML_Reg.MLReg_prioritize_compounds(
    ori_data=enum114_df,
    model='output/ML/trained_models/20260513/PCSK9_RF_H236.joblib',
    top=300,
    features_n=None,        # use the bundle's saved featurizer name
    verbose=True,
)

In [ ]:
## write sdf in dropbox folder:
if pred114_df.shape[0] > 0:
    n = rdkit_tools.write_filtered_enum_sdf(
        pred114_df,
        source_dir='data/enumeration/20260427/', # exact path to sdf file which let's copy verbatim the stereochemistry and attributes
        out_path=DROPBOX_ML+'predictions/20260518_Enamine_114k_pred.sdf',
        pred_col='predicted_label',
)

#### 20260608 Re-Prioritize these compounds based on curated data
We remove all compounds that are found to be FBX indepent downmodulation as well as not validation in n=2

In [3]:
## get 114 virtual enumerations:
enum750k_df = pd.read_csv('data/enumeration/20260608_PCSK9/Serac scaffolds broad enumeration.csv', sep='\t',).rename(columns={'SMILES':'smiles'})
enum750k_df['compound'] = 'X'+enum750k_df.index.astype(str)
enum750k_df.head(1)

,smiles,RSN,compound
0,C#CC#CC(C)(C)NC(=O)COc1ccc2c(c1)[C@@H]1CNC(=O)...,m11____20028868____117735006,X0


In [ ]:
%%time 
## get 114 virtual enumerations:
enum750k_df = pd.read_csv('data/enumeration/20260608_PCSK9/Serac scaffolds broad enumeration.csv', sep='\t',).rename(columns={'SMILES':'smiles'})
enum750k_df['compound'] = 'X'+enum750k_df.index.astype(str)

# # enum114_df['smiles'] = enum114_df['smiles'].progress_apply(lambda x: rdkit_tools.convert_smiles_to_canonical(x))

# ## prioritize compounds:
# pred114_df = ML_Reg.MLReg_prioritize_compounds(
#     ori_data=enum750k_df,
#     # model='output/ML/trained_models/20260513/PCSK9_RF_H236.joblib',
#     model='output/ML/trained_models/20260608_PCSK9/PCSK9_RF_H236.joblib',
#     top=1000,
#     features_n=None,        # use the bundle's saved featurizer name
#     verbose=True,
# )

> loading bundle: gene=PCSK9  R²=0.147  features=H236  (2529 cols)
> featurising 758,691 unique compounds via compute_H236_features...


predicting: 100%|██████████| 21/21 [00:08<00:00,  2.41chunk/s]


> PCSK9  R²=0.147  features=H236  (2529 cols)  predicted 758,691 unique / 758,691 rows; top 1000
CPU times: user 32min 46s, sys: 21.6 s, total: 33min 8s
Wall time: 1h 1min 9s


In [79]:
pred114_df.head(1)

,smiles,RSN,compound,predicted_label
0,COC(=O)N1[C@H]2Cc3ccc(-c4ccnc(NCC5CCN(c6cc(Cl)...,m27____2757582____117711006,X305996,-1.187251


In [82]:
## write sdf in dropbox folder (SMILES-only, no template):
## tag_cols = the fields to embed as SDF tags (one each). 'compound' becomes the
## molecule title automatically. pred_col is just the single-tag shortcut for
## TEMPLATE mode; in SMILES-only mode use tag_cols for one-or-many fields.
if pred114_df.shape[0] > 0:
    n = rdkit_tools.write_filtered_enum_sdf(
        pred114_df,
        out_path=DROPBOX_ML+'20260608_Echoes_PCSK9_750k_enamine.sdf',
        tag_cols=['RSN', 'predicted_label'],
    )
    print(f'wrote {n} records')


wrote 1000 records
